In [1]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [3]:

eurusd = pd.read_csv("common/MachineLearningModel/output/fifteen_mins/EURUSD_15_Min.csv")
eurjpy = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURJPY_15_Min.csv')
eurcad = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURCAD_15_Min.csv')
euraud = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURAUD_15_Min.csv')
eurgbp = pd.read_csv('common/MachineLearningModel/output/fifteen_mins/EURGBP_15_Min.csv')

In [4]:
from ta.trend import macd,cci,adx,macd_signal,adx_pos,adx_neg
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
def calculate(pd: pd.DataFrame):
    pdrsi = rsi(pd['close'],14)
    # rsi.dropna(axis=0,inplace=True)
    pdcci = cci(pd['high'],pd['low'],pd['close'],14)
    # cci.dropna(axis=0,inplace=True)
    pdadx = adx(pd['high'],pd['low'],pd['close'])
    pdadx_pos = adx_pos(pd['high'],pd['low'],pd['low']) 
    pdadx_neg = adx_neg(pd['high'],pd['low'],pd['low'])
    # adx.dropna(axis=0,inplace=True)
    pdmacd = macd(pd['close'])
    # macd.dropna(axis=0,inplace=True)
    pdmacd_signal = macd_signal(pd['close'])
    # macd_signal.dropna(axis=0,inplace=True)
    pdstochrsi_d = stochrsi_d(pd['close'])
    pdstochrsi_k = stochrsi_k(pd['close'])
    pdstochrsi = stochrsi(pd['close'])
    # stochrsi.dropna(axis=0,inplace=True)
    pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
    pd2['rsi'] = pdrsi
    pd2['cci'] = pdcci
    pd2['adx'] = pdadx
    pd2['adx_pos'] = pdadx_pos
    pd2['adx_neg'] = pdadx_neg
    pd2['macd'] = pdmacd
    pd2['macd_signal'] = pdmacd_signal
    pd2['stochrsi_d'] = pdstochrsi_d
    pd2['stochrsi_k'] = pdstochrsi_k
    pd2['stochrsi'] = pdstochrsi
    return pd2


In [5]:
pd1 = calculate(eurusd)
pd2 = calculate(eurjpy)
pd3 = calculate(eurcad)
pd4 = calculate(euraud)
pd5 = calculate(eurgbp)

In [6]:
data = pd.concat([pd1,pd2,pd3,pd4,pd5])
data.dropna(axis=0,inplace=True)
# print(data.columns)
# print(data.count())
print(data.tail(-1))

         symbol     open     high      low    close  volume        rsi  \
34    FX:EURUSD  1.09692  1.09694  1.09613  1.09655  2215.0  21.530159   
35    FX:EURUSD  1.09655  1.09681  1.09580  1.09607  2717.0  20.154702   
36    FX:EURUSD  1.09607  1.09634  1.09535  1.09538  3207.0  18.340813   
37    FX:EURUSD  1.09538  1.09586  1.09532  1.09578  3561.0  22.684860   
38    FX:EURUSD  1.09578  1.09622  1.09517  1.09580  3702.0  22.905694   
...         ...      ...      ...      ...      ...     ...        ...   
5991  FX:EURGBP  0.85497  0.85551  0.85492  0.85497  3274.0  36.735718   
5992  FX:EURGBP  0.85497  0.85513  0.85477  0.85493  4254.0  35.969425   
5993  FX:EURGBP  0.85493  0.85524  0.85486  0.85503  2245.0  39.374200   
5994  FX:EURGBP  0.85503  0.85507  0.85481  0.85498  2528.0  38.278208   
5995  FX:EURGBP  0.85498  0.85501  0.85478  0.85488   515.0  36.113122   

             cci        adx    adx_pos    adx_neg      macd  macd_signal  \
34   -114.775738  46.664536   7.805

In [7]:
data['RSI_1'] = np.where(data['rsi'] < 30, 1, np.where(data['rsi'] > 70, 2, 0))
# data['MACD_1'] = np.where(data['macd'] < data['macd_signal'], 2, np.where(data['macd'] > data['macd_signal'], 1, 0))
# data['CCI_1'] = np.where(data['cci'] < -80, 1, np.where(data['cci'] > 80, 2, 0))
adx_condition = (data['adx'] > 25.00) | (data['rsi'] > 70)
adx_condition_2 = (data['adx'] > 25.00) | (data['rsi'] < 30)
data['ADX_1'] = np.where((data['adx'] > 25.00) & (data['adx_pos'] < data['adx_neg']), 1, np.where((data['adx'] > 25.00) & (data['adx_pos'] > data['adx_neg']), 2, 0))
conditions_3 = (data['stochrsi'] > 0.75) & (data['stochrsi_k'] < data['stochrsi_d'])
conditions_4 = (data['stochrsi'] < 0.25) & (data['stochrsi_k'] > data['stochrsi_d'])
data['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
conditions_2 = (data['ADX_1'] == 2)
conditions_1 = (data['ADX_1'] == 1)
data['Prediction'] = np.where(adx_condition_2 & (data['open'] < data['close']), 1,
                              np.where(adx_condition & (data['open'] > data['close']), 2, 0))


In [8]:
# data['rsi'] = data['rsi'].astype(dtype=int)
# data['cci'] = data['cci'].astype(dtype=int)
# data['adx'] = data['adx'].astype(dtype=int)
# data['adx_pos'] = data['adx_pos'].astype(dtype=int)
# data['adx_neg'] = data['adx_neg'].astype(dtype=int)

In [9]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
# data.drop(axis=1,labels=['RSI_1'],inplace=True)

In [10]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


      symbol     open     high      low    close  volume        rsi  \
0  FX:EURUSD  1.09691  1.09719  1.09679  1.09692  2601.0  22.635875   
1  FX:EURUSD  1.09692  1.09694  1.09613  1.09655  2215.0  21.530159   
2  FX:EURUSD  1.09655  1.09681  1.09580  1.09607  2717.0  20.154702   
3  FX:EURUSD  1.09607  1.09634  1.09535  1.09538  3207.0  18.340813   
4  FX:EURUSD  1.09538  1.09586  1.09532  1.09578  3561.0  22.684860   

          cci        adx   adx_pos    adx_neg      macd  macd_signal  \
0 -125.095024  44.853912  8.295545  42.273287 -0.001531    -0.001047   
1 -114.775738  46.664536  7.805919  44.587462 -0.001634    -0.001165   
2 -106.059162  48.446073  7.232676  43.712518 -0.001734    -0.001279   
3 -106.771273  50.231559  6.712336  43.837845 -0.001848    -0.001392   
4  -96.416773  51.898280  6.440172  42.285625 -0.001885    -0.001491   

   stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  Prediction  
0    0.000359    0.001077  0.003232      1      1          1     

In [11]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[0 1 2]


In [12]:
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.ensemble import StackingClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.datasets import make_classification
# # Generate a synthetic binary classification dataset
# X = data.iloc[:,6:-1]
# y = data.iloc[:, -1]

# # Split the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# # Define the base learners
# base_learners = [
#     ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
#     ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
#     ('xgb', XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2))
# ]
# # Define the meta-learner
# meta_learner = LogisticRegression()
# # Build the Stacking classifier
# stacking_clf = StackingClassifier(estimators=base_learners, final_estimator=meta_learner)
# # Train the Stacking classifier
# stacking_clf.fit(X_train, y_train)
# # Evaluate the model
# stacking_clf.score(X_test, y_test)

In [13]:
# import pickle
# combine_final_model = pickle.dump(stacking_clf, open('combineclassifier.sav','wb'))

In [14]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



Prediction
0    18964
1     5514
2     5336
Name: count, dtype: int64
Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'adx_pos', 'adx_neg', 'macd', 'macd_signal', 'stochrsi_d', 'stochrsi_k',
       'stochrsi', 'RSI_1', 'ADX_1', 'STOCH.RSI', 'Prediction'],
      dtype='object')
         rsi         cci        adx   adx_pos    adx_neg      macd  \
0  22.635875 -125.095024  44.853912  8.295545  42.273287 -0.001531   
1  21.530159 -114.775738  46.664536  7.805919  44.587462 -0.001634   
2  20.154702 -106.059162  48.446073  7.232676  43.712518 -0.001734   
3  18.340813 -106.771273  50.231559  6.712336  43.837845 -0.001848   
4  22.684860  -96.416773  51.898280  6.440172  42.285625 -0.001885   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  
0    -0.001047    0.000359    0.001077  0.003232      1      1          1  
1    -0.001165    0.000718    0.001077  0.000000      1      1          1  
2    -0.001279    0.001077    0.00

In [15]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 99.99161460735398
Accuracy on test data by XGBoost Classifier: 95.33791715579406


In [16]:
final_xgb_model = XGBClassifier()
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [17]:
import pickle
# xgb_final_model = pickle.dump(final_xgb_model, open('xgbclassifier.sav','wb'))

In [18]:
from TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')


error while signin
you are using nologin method, data you access may be limited


In [20]:
import random
# symbols = random.choice(['EURUSD','EURJPY','GBPUSD','EURGBP'])
symbols = 'EURJPY'
print(symbols)
response_data = s.get_hist(symbol=symbols,exchange='FX',interval=main.Interval.in_15_minute,n_bars=150,extended_session=False)
pd3 = calculate(response_data)
pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3['ADX_1'] = np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] < pd3['adx_neg']), 1, np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] > pd3['adx_neg']), 2, 0))
conditions_3 = (pd3['stochrsi'] > 0.75) & (pd3['stochrsi_k'] < pd3['stochrsi_d'])
conditions_4 = (pd3['stochrsi'] < 0.25) & (pd3['stochrsi_k'] > pd3['stochrsi_d'])
pd3['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
# pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3.dropna(inplace=True)
pd3.reset_index()
print(pd3.iloc[-1])
data_1 = pd3.iloc[-1,5:].to_dict()
data_1 = pd.DataFrame({key: [value] for key, value in data_1.items()})
print(data_1)
output = final_xgb_model.predict(pd.DataFrame(data_1))
print(output)

EURJPY
open            163.895000
high            164.054000
low             163.769000
close           164.054000
volume         2582.000000
rsi              82.791172
cci             144.700973
adx              43.695883
adx_pos          26.218085
adx_neg          11.741491
macd              0.141510
macd_signal       0.103276
stochrsi_d        0.972701
stochrsi_k        0.969203
stochrsi          1.000000
RSI_1             2.000000
ADX_1             2.000000
STOCH.RSI         2.000000
Name: 2024-04-03 10:00:00, dtype: float64
         rsi         cci        adx    adx_pos    adx_neg     macd  \
0  82.791172  144.700973  43.695883  26.218085  11.741491  0.14151   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  
0     0.103276    0.972701    0.969203       1.0    2.0    2.0        2.0  
[1]
